# Глава 27. Несбалансированные данные

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь распознать проблему дисбаланса классов, объяснить, почему accuracy тут вводит в заблуждение, и применить class_weight и балансировку для её решения.

В учебных датасетах классы часто примерно равны. В реальности — почти никогда. Мошеннических операций — доли процента, редких болезней — единицы на тысячу, спама — меньшинство. Такие данные называют несбалансированными, и обычные подходы на них ломаются. Это один из самых частых реальных сценариев.

**Аналогия: иголка в стоге сена  💡**

Найти мошенническую операцию среди миллиона честных — как иголку в стоге сена. Модель, которая просто говорит «всё честное», будет права в 99,9% случаев (высокая accuracy!), но не найдёт ни одной иголки — ради которой всё и затевалось. Дисбаланс — это когда важное встречается редко, и именно его нельзя пропустить.

## Ловушка accuracy

Мы уже касались этого в главе 11, теперь разберём вплотную. При дисбалансе 99% на 1% модель «всегда предсказывай большинство» даёт accuracy 99% — и полностью бесполезна. Поэтому на несбалансированных данных accuracy запрещена как главная метрика.

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

# Представим сильный дисбаланс: 990 честных, 10 мошенников
y_true = np.array([0]*990 + [1]*10)
y_pred = np.array([0]*1000)   # модель "всё честное"

print("Accuracy:", accuracy_score(y_true, y_pred))   # 0.99 — обман!
print("Recall:", recall_score(y_true, y_pred))       # 0.0 — правда

Accuracy = 0,99 выглядит прекрасно, но recall = 0,0 говорит правду: модель не поймала ни одного мошенника. Recall (полнота) — вот честная метрика для дисбаланса: какую долю редкого класса модель реально нашла.

## Решение 1: веса классов (class_weight)

Самый простой и часто лучший приём — сказать модели, что ошибка на редком классе «дороже». Параметр class_weight="balanced" автоматически повышает важность редкого класса при обучении. Модель начинает внимательнее относиться к меньшинству.

In [ ]:
# === Подготовка: искусственный несбалансированный набор (95% / 5%) ===
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=2000, weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

from sklearn.ensemble import RandomForestClassifier

# class_weight="balanced" — редкий класс становится "важнее"
model = RandomForestClassifier(class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

## Решение 2: балансировка данных

Можно выровнять сами данные. Undersampling — выбросить часть большого класса. Oversampling — размножить редкий класс. Умный вариант oversampling — SMOTE: он не копирует редкие примеры, а создаёт новые похожие «синтетические» между существующими.

In [ ]:
# SMOTE из библиотеки imbalanced-learn
# Запускать в Colab: !pip install imbalanced-learn -q
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_bal, y_bal = sm.fit_resample(X_train, y_train)
# Теперь классы в X_bal примерно равны

**Внимание: балансируй только train  ⚠️**

Применяй SMOTE и любую балансировку ТОЛЬКО к обучающим данным, никогда к тесту. Тест должен отражать реальное соотношение классов, иначе оценка будет ложной. Балансировка теста — это разновидность утечки данных из главы 6.

## Локальный пример: обнаружение мошенничества

Банк в Таджикистане ищет мошеннические переводы среди миллионов честных. Их доля — сотые доли процента. Здесь дороже всего пропустить мошенника (recall важнее precision), а accuracy бесполезна. Правильный подход: class_weight или SMOTE на train, метрика recall (или F1), проверка на реальном несбалансированном тесте.

## Задания

Базовый уровень

Создай искусственный несбалансированный набор (например, 95% на 5%). Обучи модель без class_weight и с class_weight="balanced". Сравни recall для редкого класса — насколько вырос?

Со звёздочкой  ⭐

На несбалансированном датасете (найди на Kaggle "credit card fraud" или сделай сам) сравни три подхода: обычная модель, class_weight, SMOTE. Сравнивай по recall и F1, НЕ по accuracy. Какой подход поймал больше редкого класса? Балансировку применяй только к train.

Вопросы для обсуждения  💬

Почему на несбалансированных данных accuracy запрещена как главная метрика?

Чем SMOTE лучше простого копирования редких примеров?

Почему балансировать можно только train, но не test?

**Проверь себя  ℹ️**

1. Почему модель с accuracy 99% может быть бесполезной?

2. Что делает class_weight="balanced"?

3. Почему нельзя применять SMOTE к тестовым данным?